# Processing NOAA GOES into website artifacts

This notebook turns **NOAA GOES-16** satellite data (accessed anonymously from the
[NOAA Open Data on AWS registry](https://registry.opendata.aws/noaa-goes/)) into the small,
committed files the website loads. **The website never reads NetCDF, never hits S3, and needs
no Python at runtime** — it only loads the JSON/WebP this notebook writes into `data/` and `assets/`.

Outputs:
- `data/goes_sst_grid_hurricane.json`, `data/goes_sst_grid_offseason.json` — Viz 2 seasonal SST grids (Sep vs Feb 2022)
- `assets/goes_sst_layer_hurricane.webp`, `assets/goes_sst_layer_offseason.webp` — rendered fallbacks
- `assets/goes_storm_visible.webp`, `goes_storm_ir.webp`, `goes_water_vapor.webp` — Hurricane Ian layers (Viz 3)
- `assets/goes_storm_sst.webp`, `data/goes_storm_sst.json` — sea surface temp in the storm region
- `data/goes_metadata.json` — records what was produced (drives labels/tooltips and keeps the site flexible)

Source products: `ABI-L2-SSTF` (full-disk SST) and `ABI-L2-MCMIPC` (CONUS multi-band cloud & moisture imagery, which carries visible, clean-IR, and water-vapor bands in one file).

In [ ]:
# One-time install (skip if already present)
# !pip install --upgrade xarray s3fs netCDF4 zarr h5netcdf h5py pyproj matplotlib pillow

## Setup

In [1]:
import datetime as dt, json, os
import numpy as np
import s3fs, xarray as xr
from pyproj import CRS, Transformer
import matplotlib
matplotlib.use("Agg")
import matplotlib.cm as cm
from PIL import Image

fs = s3fs.S3FileSystem(anon=True)            # anonymous (no AWS account needed)
os.makedirs("data", exist_ok=True)
os.makedirs("assets", exist_ok=True)
CACHE = "/tmp/goes_cache"; os.makedirs(CACHE, exist_ok=True)

def doy(y, m, d):
    return (dt.date(y, m, d) - dt.date(y, 1, 1)).days + 1

def fetch_local(s3path):
    """Download to local cache (fast sequential) instead of slow remote random reads."""
    local = os.path.join(CACHE, s3path.split("/")[-1])
    if not os.path.exists(local):
        print("  downloading", s3path.split("/")[-1][:46], "...")
        fs.get(s3path, local)
    return local

def open_local(s3path):
    return xr.open_dataset(fetch_local(s3path), engine="h5netcdf")

## Preflight

Before any heavy processing, confirm access, product availability, file sizes, that a sample file
opens, and that the geostationary -> lat/lon reprojection produces sane values. Then choose the
best storm/date/product combination. Prints a clear status summary.

In [2]:
def doy_(y,m,d): return doy(y,m,d)

summary = dict(goes_access="fail", sst="no", storm="no", water_vapor="no",
               reprojection="untested", storm_choice=None, ts=None)
try:
    fs.ls("noaa-goes16/ABI-L2-SSTF/2022/263/12/")
    summary["goes_access"] = "success"
except Exception as e:
    print("S3 access failed:", e)

# SST availability
sst_files = fs.glob("s3://noaa-goes16/ABI-L2-SSTF/2022/263/12/*.nc")
if sst_files:
    summary["sst"] = "yes"
    print("SST sample %.1f MB" % (fs.info(sst_files[0])["size"]/1e6))

# Storm imagery / water vapor availability (try candidate storms)
candidates = [("Hurricane Ian (2022)",2022,9,28,17),("Hurricane Ida (2021)",2021,8,29,16),
              ("Hurricane Laura (2020)",2020,8,26,18),("Hurricane Dorian (2019)",2019,9,1,16)]
for label,y,m,d,h in candidates:
    f = fs.glob(f"s3://noaa-goes16/ABI-L2-MCMIPC/{y}/{doy(y,m,d):03d}/{h:02d}/*.nc")
    if f:
        summary.update(storm="yes", water_vapor="yes", storm_choice=label,
                       ts=f"{y}-{m:02d}-{d:02d} {h:02d}:00 UTC")
        print(f"Storm imagery {label}: {len(f)} files, %.1f MB each" % (fs.info(f[0])["size"]/1e6))
        break

# Reprojection sanity
test = sst_files[0] if sst_files else None
if test:
    ds = open_local(test); p = ds["goes_imager_projection"]
    H = float(p.attrs["perspective_point_height"]); lon0 = float(p.attrs["longitude_of_projection_origin"])
    crs = CRS.from_proj4(f"+proj=geos +h={H} +lon_0={lon0} +a={p.attrs['semi_major_axis']} "
                         f"+b={p.attrs['semi_minor_axis']} +sweep={p.attrs.get('sweep_angle_axis','x')} +units=m +no_defs")
    t = Transformer.from_crs(crs, "EPSG:4326", always_xy=True)
    lon, lat = t.transform(float(ds.x[len(ds.x)//2])*H, float(ds.y[len(ds.y)//2])*H)
    summary["reprojection"] = "success" if (-180<=lon<=180 and -90<=lat<=90) else "fallback needed"
    print(f"reprojection check -> lon={lon:.2f}, lat={lat:.2f}"); ds.close()

print("\n================ PREFLIGHT SUMMARY ================")
print("GOES access:", summary["goes_access"])
print("SST product available:", summary["sst"])
print("Storm imagery available:", summary["storm"])
print("Water vapor product available:", summary["water_vapor"])
print("Reprojection status:", summary["reprojection"])
print("Selected storm:", summary["storm_choice"])
print("Selected timestamps:", summary["ts"])
print("Export strategy:", "both" if summary["sst"]=="yes" and summary["storm"]=="yes" else "PNG layers / JSON grid")
print("==================================================")

SST sample 30.5 MB
Storm imagery Hurricane Ian (2022): 12 files, 67.4 MB each
reprojection check -> lon=-74.99, lat=-0.01

================ PREFLIGHT SUMMARY ================
GOES access: success
SST product available: yes
Storm imagery available: yes
Water vapor product available: yes
Reprojection status: success
Selected storm: Hurricane Ian (2022)
Selected timestamps: 2022-09-28 17:00 UTC
Export strategy: both


## Configuration

Chosen from preflight: **Hurricane Ian (2022-09-28 17 UTC)** for the storm layers, and a clear-ish
pre-storm day (2022-09-20) for the sea-surface-temperature composite (the surface is cloud-obscured
under an active storm, so a recent clear-sky composite best shows the warm water in the path).

In [3]:
STORM = dict(name="Hurricane Ian", year_label="2022", date_utc="2022-09-28 17:00 UTC",
             category_at_view="Category 4 / 5",
             landfall=dict(name="Cayo Costa, SW Florida", lat=26.7, lon=-82.3),
             blurb="One of the costliest U.S. hurricanes on record; made landfall in southwest Florida on Sep 28, 2022.")
STORM_DT = (2022, 9, 28, 17)
SST_DAY  = (2022, 9, 20); SST_HOURS = [12, 15, 17]
BROAD_BOX = dict(lon0=-98.0, lon1=-15.0, lat0=7.0, lat1=33.0)   # Viz 2
STORM_BOX = dict(lon0=-92.0, lon1=-76.0, lat0=20.0, lat1=32.0)  # Viz 3

meta = dict(generated=dt.datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC"),
            source="NOAA GOES-16 (GOES-East), accessed via NOAA Open Data on AWS",
            products={"sst": "ABI-L2-SSTF", "imagery": "ABI-L2-MCMIPC"},
            storm=STORM, reprojection="success", artifacts={},
            boxes={"broad": BROAD_BOX, "storm": STORM_BOX},
            notes=["SST shown is a clear-sky composite from a recent peak-season day; under an active storm the surface is cloud-obscured.",
                   "GOES water-vapor imagery depicts atmospheric moisture, not measured precipitation."])

## Helper functions (reprojection, binning, rendering)

In [4]:
def get_proj(ds):
    p = ds["goes_imager_projection"]
    H = float(p.attrs["perspective_point_height"]); lon0 = float(p.attrs["longitude_of_projection_origin"])
    a = float(p.attrs["semi_major_axis"]); b = float(p.attrs["semi_minor_axis"]); sweep = p.attrs.get("sweep_angle_axis","x")
    return H, CRS.from_proj4(f"+proj=geos +h={H} +lon_0={lon0} +a={a} +b={b} +sweep={sweep} +units=m +no_defs")

def xy_bounds_for_box(H, crs, box, pad=0.5):
    fwd = Transformer.from_crs("EPSG:4326", crs, always_xy=True)
    LO, LA = np.meshgrid(np.linspace(box["lon0"]-pad, box["lon1"]+pad, 25),
                         np.linspace(box["lat0"]-pad, box["lat1"]+pad, 25))
    X, Y = fwd.transform(LO.ravel(), LA.ravel())
    X = X[np.isfinite(X)]/H; Y = Y[np.isfinite(Y)]/H
    return X.min(), X.max(), Y.min(), Y.max()

def slice_xy(ds, xr0, xr1, yr0, yr1):
    xv, yv = ds["x"].values, ds["y"].values
    return ds.sel(x=slice(xr0,xr1) if xv[0]<xv[-1] else slice(xr1,xr0),
                  y=slice(yr0,yr1) if yv[0]<yv[-1] else slice(yr1,yr0))

def latlon_grid(sub, H, crs, stride=1):
    inv = Transformer.from_crs(crs, "EPSG:4326", always_xy=True)
    XX, YY = np.meshgrid(sub["x"].values[::stride]*H, sub["y"].values[::stride]*H)
    return inv.transform(XX, YY)

def bin_to_grid(lon, lat, vals, box, nx):
    span_lon = box["lon1"]-box["lon0"]; span_lat = box["lat1"]-box["lat0"]
    ny = max(20, int(round(nx*span_lat/span_lon)))
    le = np.linspace(box["lon0"], box["lon1"], nx+1); ae = np.linspace(box["lat0"], box["lat1"], ny+1)
    lon, lat, vals = lon.ravel(), lat.ravel(), vals.ravel()
    m = np.isfinite(lon)&np.isfinite(lat)&np.isfinite(vals); lon,lat,vals = lon[m],lat[m],vals[m]
    ix = np.clip(np.digitize(lon, le)-1, 0, nx-1); iy = np.clip(np.digitize(lat, ae)-1, 0, ny-1)
    sums = np.zeros((ny,nx)); cnts = np.zeros((ny,nx))
    np.add.at(sums,(iy,ix),vals); np.add.at(cnts,(iy,ix),1)
    with np.errstate(invalid="ignore"):
        return np.where(cnts>0, sums/cnts, np.nan), ny

def render_webp(arr, cmap_name, vmin, vmax, path, nan_transparent=True, gamma=1.0, flipud=False):
    a = arr.astype(float); norm = np.clip((a-vmin)/(vmax-vmin), 0, 1)
    if gamma != 1.0: norm = np.power(norm, gamma)
    rgba = (cm.get_cmap(cmap_name)(norm)*255).astype(np.uint8)
    if nan_transparent: rgba[~np.isfinite(a), 3] = 0
    if flipud: rgba = rgba[::-1]
    Image.fromarray(rgba, "RGBA").save(path, "WEBP", quality=88, method=6)

## Viz 2 — Atlantic sea surface temperature grid

Composite a few hours of full-disk SST over the Gulf/Caribbean/tropical-Atlantic box, mask by the
data-quality flag, convert Kelvin to Celsius, reproject to lat/lon, and bin to a ~200x63 grid.

In [5]:
y, m, d = SST_DAY
sums = cnts = lon = lat = None; H = crs = None; ref = None; xr0=xr1=yr0=yr1=None
for h in SST_HOURS:
    files = fs.glob(f"s3://noaa-goes16/ABI-L2-SSTF/{y}/{doy(y,m,d):03d}/{h:02d}/*.nc")
    if not files: continue
    ds = open_local(files[0])
    if H is None:
        H, crs = get_proj(ds); xr0,xr1,yr0,yr1 = xy_bounds_for_box(H, crs, BROAD_BOX)
    sub = slice_xy(ds, xr0,xr1,yr0,yr1); S = 3
    sst = np.where(sub["DQF"].values[::S,::S]==0, sub["SST"].values[::S,::S], np.nan) - 273.15
    if lon is None:
        lon, lat = latlon_grid(sub, H, crs, stride=S); ref = sst.shape
        lon, lat = lon[:ref[0],:ref[1]], lat[:ref[0],:ref[1]]; sums = np.zeros(ref); cnts = np.zeros(ref)
    sst = sst[:ref[0],:ref[1]]; g = np.isfinite(sst); sums[g]+=sst[g]; cnts[g]+=1; ds.close()
with np.errstate(invalid="ignore"):
    comp = np.where(cnts>0, sums/cnts, np.nan)
grid, ny = bin_to_grid(lon, lat, comp, BROAD_BOX, nx=200); nx = grid.shape[1]
vmin, vmax = float(np.nanpercentile(grid,2)), float(np.nanpercentile(grid,98))
flat = [None if not np.isfinite(v) else round(float(v),2) for v in grid[::-1].ravel()]   # row0 = north
json.dump(dict(nx=nx, ny=ny, lon0=BROAD_BOX["lon0"], lon1=BROAD_BOX["lon1"],
               lat0=BROAD_BOX["lat1"], lat1=BROAD_BOX["lat0"], vmin=round(vmin,2), vmax=round(vmax,2),
               unit="degC", values=flat), open("data/goes_sst_grid.json","w"))
render_webp(grid[::-1], "inferno", vmin, vmax, "assets/goes_sst_layer.webp")
meta["artifacts"].update(sst_grid="data/goes_sst_grid.json", sst_layer="assets/goes_sst_layer.webp")
meta["sst"] = dict(vmin=round(vmin,2), vmax=round(vmax,2), day=f"{y}-{m:02d}-{d:02d}")
print(f"SST grid {nx}x{ny}, range {vmin:.1f}..{vmax:.1f} C")

SST grid 200x63, range 25.3..30.9 C


C:\Users\KateM\AppData\Local\Temp\ipykernel_58512\1032458481.py:40: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  rgba = (cm.get_cmap(cmap_name)(norm)*255).astype(np.uint8)


## Viz 3 — Hurricane Ian layers (visible, clean IR, water vapor) + storm-region SST

In [6]:
y, m, d, h = STORM_DT
files = sorted(fs.glob(f"s3://noaa-goes16/ABI-L2-MCMIPC/{y}/{doy(y,m,d):03d}/{h:02d}/*.nc"))
ds = open_local(files[0]); H, crs = get_proj(ds)
bx0,bx1,by0,by1 = xy_bounds_for_box(H, crs, STORM_BOX, pad=0.0)
sub = slice_xy(ds, bx0,bx1,by0,by1)
flip = sub["y"].values[0] < sub["y"].values[-1]
c02, c13, c08 = sub["CMI_C02"].values, sub["CMI_C13"].values, sub["CMI_C08"].values
render_webp(np.clip(c02,0,1), "gray", 0.0, 0.85, "assets/goes_storm_visible.webp", nan_transparent=False, gamma=0.7, flipud=flip)
render_webp(300.0-np.clip(c13,185,300), "inferno", 0.0, 115.0, "assets/goes_storm_ir.webp", nan_transparent=False, flipud=flip)
render_webp(265.0-np.clip(c08,200,265), "GnBu", 0.0, 65.0, "assets/goes_water_vapor.webp", nan_transparent=False, flipud=flip)
ih, iw = c13.shape
meta["artifacts"].update(storm_visible="assets/goes_storm_visible.webp", storm_ir="assets/goes_storm_ir.webp",
                         water_vapor="assets/goes_water_vapor.webp")
meta["storm_image"] = dict(w=int(iw), h=int(ih), box=STORM_BOX); ds.close()
print(f"storm layers {iw}x{ih}")

C:\Users\KateM\AppData\Local\Temp\ipykernel_58512\1032458481.py:40: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  rgba = (cm.get_cmap(cmap_name)(norm)*255).astype(np.uint8)


storm layers 811x567


In [7]:
# Storm-region SST (warm water in the path), from the clear composite day, same box
y, m, d = SST_DAY
sums = cnts = lon = lat = None; Hs = crss = None; ref = None; bx0=bx1=by0=by1=None
for h in SST_HOURS:
    ff = fs.glob(f"s3://noaa-goes16/ABI-L2-SSTF/{y}/{doy(y,m,d):03d}/{h:02d}/*.nc")
    if not ff: continue
    ds = open_local(ff[0])
    if Hs is None:
        Hs, crss = get_proj(ds); bx0,bx1,by0,by1 = xy_bounds_for_box(Hs, crss, STORM_BOX, pad=0.0)
    sub = slice_xy(ds, bx0,bx1,by0,by1); S = 2
    sst = np.where(sub["DQF"].values[::S,::S]==0, sub["SST"].values[::S,::S], np.nan) - 273.15
    if lon is None:
        lon, lat = latlon_grid(sub, Hs, crss, stride=S); ref = sst.shape
        lon, lat = lon[:ref[0],:ref[1]], lat[:ref[0],:ref[1]]; sums = np.zeros(ref); cnts = np.zeros(ref)
    sst = sst[:ref[0],:ref[1]]; g = np.isfinite(sst); sums[g]+=sst[g]; cnts[g]+=1; ds.close()
with np.errstate(invalid="ignore"):
    comp = np.where(cnts>0, sums/cnts, np.nan)
sgrid, sny = bin_to_grid(lon, lat, comp, STORM_BOX, nx=120); snx = sgrid.shape[1]
svmin, svmax = float(np.nanpercentile(sgrid,5)), float(np.nanpercentile(sgrid,95))
render_webp(sgrid[::-1], "inferno", svmin, svmax, "assets/goes_storm_sst.webp")
flat = [None if not np.isfinite(v) else round(float(v),2) for v in sgrid[::-1].ravel()]
json.dump(dict(nx=snx, ny=sny, box=STORM_BOX, vmin=round(svmin,2), vmax=round(svmax,2),
               unit="degC", values=flat), open("data/goes_storm_sst.json","w"))
meta["artifacts"].update(storm_sst_layer="assets/goes_storm_sst.webp", storm_sst_grid="data/goes_storm_sst.json")
meta["storm_sst"] = dict(vmin=round(svmin,2), vmax=round(svmax,2))
print(f"storm SST {snx}x{sny}, range {svmin:.1f}..{svmax:.1f} C")

storm SST 120x90, range 29.0..31.0 C


C:\Users\KateM\AppData\Local\Temp\ipykernel_58512\1032458481.py:40: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  rgba = (cm.get_cmap(cmap_name)(norm)*255).astype(np.uint8)


In [8]:
import urllib.request
import os

# Download ERSST v5 monthly file for 1980
# Format: ersst.v5.YYYYMM.nc
url = "https://www.ncei.noaa.gov/pub/data/cmb/ersst/v5/netcdf/ersst.v5.198009.nc"
out = "ersst.v5.198009.nc"

if not os.path.exists(out):
    print("Downloading ERSST 1980-09...")
    urllib.request.urlretrieve(url, out)
    print("Done")
else:
    print("Already downloaded")

Already downloaded


In [9]:
import xarray as xr
import numpy as np
import json

ersst = xr.open_dataset('ersst.v5.198009.nc')
print(ersst)  # check variable names

<xarray.Dataset> Size: 130kB
Dimensions:  (time: 1, lev: 1, lat: 89, lon: 180)
Coordinates:
  * time     (time) object 8B 1980-09-01 00:00:00
  * lev      (lev) float64 8B 0.0
  * lat      (lat) float64 712B -88.0 -86.0 -84.0 -82.0 ... 82.0 84.0 86.0 88.0
  * lon      (lon) float64 1kB 0.0 2.0 4.0 6.0 8.0 ... 352.0 354.0 356.0 358.0
Data variables:
    sst      (time, lev, lat, lon) float32 64kB ...
    ssta     (time, lev, lat, lon) float32 64kB ...
Attributes: (12/38)
    Conventions:                CF-1.6, ACDD-1.3
    metadata_link:              https://doi.org/10.7289/V5T72FNM
    id:                         ersst.v5.198009
    naming_authority:           gov.noaa.ncei
    title:                      NOAA ERSSTv5 (in situ only)
    summary:                    ERSST.v5 is developped based on v4 after revi...
    ...                         ...
    platform:                   Ship and Buoy SSTs from ICOADS R3.0 and NCEP GTS
    instrument:                 Conventional thermometers
 

In [11]:
import xarray as xr
import numpy as np
import json

# ── Load ──────────────────────────────────────────────────────────────
ersst = xr.open_dataset('ersst.v5.198009.nc')
sst = ersst.sst.squeeze()  # drop time and lev dims

# ── ERSST uses 0–360 lon, convert to -180–180 ─────────────────────────
sst = sst.assign_coords(lon=(sst.lon.values + 180) % 360 - 180)
sst = sst.sortby('lon')

# ── Crop to your GOES domain ──────────────────────────────────────────
LON0, LON1 = -98, -15
LAT0, LAT1 =   7,  33   # south, north (ERSST lat goes S→N)

sst_crop = sst.sel(
    lon=slice(LON0, LON1),
    lat=slice(LAT0, LAT1)
)

print("Cropped shape:", sst_crop.shape)
print("Lon range:", float(sst_crop.lon.min()), "to", float(sst_crop.lon.max()))
print("Lat range:", float(sst_crop.lat.min()), "to", float(sst_crop.lat.max()))
print("SST range:", float(sst_crop.min()), "to", float(sst_crop.max()))

# ── Convert to list, replace NaN (land/missing) with null ────────────
values_raw = sst_crop.values  # shape: (lat, ny, nx) after squeeze
ny, nx = values_raw.shape
values_list = []
for r in range(ny):
    for c in range(nx):
        v = float(values_raw[r, c])
        values_list.append(None if np.isnan(v) else round(v, 2))

# ── Build grid JSON matching your existing format ─────────────────────
grid = {
    "lon0": float(sst_crop.lon.min()),
    "lon1": float(sst_crop.lon.max()),
    "lat0": float(sst_crop.lat.max()),   # north (lat0 = north in your format)
    "lat1": float(sst_crop.lat.min()),   # south
    "nx": nx,
    "ny": ny,
    "vmin": 19,   # match your existing color scale
    "vmax": 31,
    "values": values_list
}

with open('data/sst_grid_1980.json', 'w') as f:
    json.dump(grid, f)

print(f"Exported: {nx}×{ny} grid → data/sst_grid_1980.json")
print(f"Non-null cells: {sum(1 for v in values_list if v is not None)}")

Cropped shape: (13, 42)
Lon range: -98.0 to -16.0
Lat range: 8.0 to 32.0
SST range: 21.83426284790039 to 29.76316261291504
Exported: 42×13 grid → data/sst_grid_1980.json
Non-null cells: 497


## Write metadata

In [12]:
json.dump(meta, open("data/goes_metadata.json","w"), indent=2)
print(json.dumps(meta["artifacts"], indent=2))

{
  "sst_grid": "data/goes_sst_grid.json",
  "sst_layer": "assets/goes_sst_layer.webp",
  "storm_visible": "assets/goes_storm_visible.webp",
  "storm_ir": "assets/goes_storm_ir.webp",
  "water_vapor": "assets/goes_water_vapor.webp",
  "storm_sst_layer": "assets/goes_storm_sst.webp",
  "storm_sst_grid": "data/goes_storm_sst.json"
}


In [13]:
import os
import glob

# Check what CMIP6 files you have locally
nc_files = glob.glob('**/*.nc', recursive=True) + glob.glob('*.nc')
for f in nc_files:
    print(f, '—', round(os.path.getsize(f)/1e6, 1), 'MB')

ersst.v5.198009.nc — 0.1 MB
data\atlantic_sst.nc — 9.6 MB
ersst.v5.198009.nc — 0.1 MB


In [14]:
import xarray as xr

ds = xr.open_dataset('data/atlantic_sst.nc')
print(ds)

<xarray.Dataset> Size: 10MB
Dimensions:  (time: 1516, lat: 31, lon: 51)
Coordinates:
  * time     (time) datetime64[ns] 12kB 1900-01-01 1900-02-01 ... 2026-04-01
  * lat      (lat) float32 124B 60.0 58.0 56.0 54.0 52.0 ... 8.0 6.0 4.0 2.0 0.0
  * lon      (lon) float32 204B 280.0 282.0 284.0 286.0 ... 376.0 378.0 380.0
Data variables:
    sst      (time, lat, lon) float32 10MB ...
Attributes: (12/39)
    climatology:               Climatology is based on 1971-2000 SST, Xue, Y....
    description:               In situ data: ICOADS2.5 before 2007 and NCEP i...
    keywords_vocabulary:       NASA Global Change Master Directory (GCMD) Sci...
    keywords:                  Earth Science > Oceans > Ocean Temperature > S...
    instrument:                Conventional thermometers
    source_comment:            SSTs were observed by conventional thermometer...
    ...                        ...
    comment:                   SSTs were observed by conventional thermometer...
    summary:      

In [24]:
def export_grid(sst_2d, filename, label):
    print(f"  export_grid called for {label}")
    
    # Check lat direction and slice accordingly
    lats_all = sst_2d.lat.values
    if lats_all[0] > lats_all[-1]:
        # Descending — slice high to low
        sst_crop = sst_2d.sel(lon=slice(LON0, LON1), lat=slice(LAT1, LAT0))
    else:
        # Ascending — slice low to high  
        sst_crop = sst_2d.sel(lon=slice(LON0, LON1), lat=slice(LAT0, LAT1))
    
    print(f"  crop shape: {sst_crop.shape}")

    lats = sst_crop.lat.values
    lons = sst_crop.lon.values
    vals = sst_crop.values

    # Ensure lats ascending for processing
    if lats[0] > lats[-1]:
        lats = lats[::-1]
        vals = vals[::-1, :]

    ny, nx = vals.shape
    values_list = []
    for r in range(ny - 1, -1, -1):  # north first
        for c in range(nx):
            v = float(vals[r, c])
            values_list.append(None if np.isnan(v) else round(v, 2))

    grid = {
        "lon0": float(lons.min()),
        "lon1": float(lons.max()),
        "lat0": float(lats.max()),
        "lat1": float(lats.min()),
        "nx": int(nx),
        "ny": int(ny),
        "vmin": 19,
        "vmax": 31,
        "values": values_list
    }
    with open(filename, 'w') as f:
        json.dump(grid, f)
    valid = vals[~np.isnan(vals)]
    print(f"{label}: {nx}×{ny}, SST {float(valid.min()):.1f}–{float(valid.max()):.1f}°C → {filename}")

In [26]:
import pandas as pd

df = pd.read_csv('data/ssp585.csv')

now_val = df[df.year.between(2018, 2023)]['projected_sst'].mean()
val_2080 = df[df.year == 2080]['projected_sst'].values[0]
offset = val_2080 - now_val

print(f"Recent average SST: {now_val:.2f}°C")
print(f"2080 projected SST: {val_2080:.2f}°C")
print(f"Offset to apply: +{offset:.2f}°C")

Recent average SST: 21.37°C
2080 projected SST: 23.64°C
Offset to apply: +2.27°C


In [27]:
import json

with open('data/goes_sst_grid_hurricane.json') as f:
    now = json.load(f)

def make_era_grid(base, offset, filename, label):
    values = [
        None if v is None else round(v + offset, 2)
        for v in base['values']
    ]
    grid = dict(base)
    grid['values'] = values
    grid['vmin'] = round(base['vmin'] + offset, 1)
    grid['vmax'] = round(base['vmax'] + offset, 1)
    with open(filename, 'w') as f:
        json.dump(grid, f)
    valid = [v for v in values if v is not None]
    print(f"{label}: SST {min(valid):.1f}–{max(valid):.1f}°C → {filename}")

make_era_grid(now,  0.0,   'data/sst_grid_now.json',  'Today')
make_era_grid(now, -1.2,   'data/sst_grid_1980.json', '1980')
make_era_grid(now,  2.27,  'data/sst_grid_2080.json', '2080')

print("Done.")

Today: SST 24.1–36.8°C → data/sst_grid_now.json
1980: SST 22.9–35.6°C → data/sst_grid_1980.json
2080: SST 26.3–39.0°C → data/sst_grid_2080.json
Done.
